# Week 1 — Food Delivery Data Analysis
## Dataset understanding and cleaning

**Purpose:** inspect the supplied dataset, identify data-quality issues, justify conservative cleaning decisions, and export a separate cleaned CSV plus a cleaning log.

### Important rules used in this notebook
- The raw CSV is never overwritten.
- Suspicious values are investigated, not automatically deleted.
- Missing values are not imputed without a defensible rule.
- Unclear category codes and labels are retained until confirmed by a data dictionary or mentor.
- Only the verified `Order_Date` representation is standardized. Mixed time columns and suspicious coordinates remain unchanged.
- This is Week 1 data understanding and cleaning, not the later exploratory analysis/report.

**Dataset limitation:** no official data dictionary was supplied. Any interpretation of coded values or unusual coordinates is therefore labelled as an observation or hypothesis, not a confirmed fact.

## 1. Imports and file locations

Place the original CSV at `data/raw/Food delivery Dataset.csv` in the project repository. Run this notebook from the `notebooks/` folder.

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Food delivery Dataset.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DOCS_DIR = PROJECT_ROOT / "docs"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {RAW_DATA_PATH}. "
        "Place the original CSV in data/raw/ before running this notebook."
    )

## 2. Load the raw data and establish a working copy

`df_raw` is the untouched source data. `df` is the working copy used for inspection and the one verified transformation. Keeping them separate makes the process reproducible and protects the original values.

In [40]:
df_raw = pd.read_csv(RAW_DATA_PATH)
df = df_raw.copy()

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
display(df.head())

Rows: 45584
Columns: 20


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken (min)
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,12-02-2022,21:55,22:10,Fog,Jam,2,Snack,motorcycle,3.0,No,Metropolitian,46
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,13-02-2022,14:55,15:05,Stormy,High,1,Meal,motorcycle,1.0,No,Metropolitian,23
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,04-03-2022,17:30,17:40,Sandstorms,Medium,1,Drinks,scooter,1.0,No,Metropolitian,21
3,0xc8b6,LUDHRES15DEL02,34.0,4.3,30.899584,75.809346,30.919584,75.829346,13-02-2022,09:20,09:30,Sandstorms,Low,0,Buffet,motorcycle,0.0,No,Metropolitian,20
4,0xdb64,KNPRES14DEL02,24.0,4.7,26.463504,80.372929,26.593504,80.502929,14-02-2022,19:50,20:05,Fog,Jam,1,Snack,scooter,1.0,No,Metropolitian,41


### 2.1 Structure and data types

`info()` shows the storage type and non-null count. `describe()` summarizes numeric columns; it does not establish whether unusual values are valid.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45584 entries, 0 to 45583
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45584 non-null  str    
 1   Delivery_person_ID           45584 non-null  str    
 2   Delivery_person_Age          43730 non-null  float64
 3   Delivery_person_Ratings      43676 non-null  float64
 4   Restaurant_latitude          45584 non-null  float64
 5   Restaurant_longitude         45584 non-null  float64
 6   Delivery_location_latitude   45584 non-null  float64
 7   Delivery_location_longitude  45584 non-null  float64
 8   Order_Date                   45584 non-null  str    
 9   Time_Orderd                  43853 non-null  str    
 10  Time_Order_picked            45584 non-null  str    
 11  Weather_conditions           44968 non-null  str    
 12  Road_traffic_density         44983 non-null  str    
 13  Vehicle_condition          

In [41]:
display(df.describe())

,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Vehicle_condition,multiple_deliveries,Time_taken (min)
count,43730.000000,43676.000000,45584.000000,45584.000000,45584.000000,45584.000000,45584.000000,44591.000000,45584.000000
mean,29.566911,4.633774,17.017948,70.229684,17.465480,70.844161,1.023385,0.744635,26.293963
std,5.815064,0.334744,8.185674,22.885575,7.335562,21.120578,0.839055,0.572510,9.384298
min,15.000000,1.000000,-30.905562,-88.366217,0.010000,0.010000,0.000000,0.000000,10.000000
25%,25.000000,4.500000,12.933284,73.170000,12.988453,73.280000,0.000000,0.000000,19.000000
50%,30.000000,4.700000,18.551440,75.897963,18.633934,76.002574,1.000000,1.000000,26.000000
75%,35.000000,4.900000,22.728163,78.044095,22.785049,78.107044,2.000000,1.000000,32.000000
max,50.000000,6.000000,30.914057,88.433452,31.054057,88.563452,3.000000,3.000000,54.000000


In [8]:
display(df.tail())

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken (min)
45579,0x7c09,JAPRES04DEL01,30.0,4.8,26.902328,75.794257,26.912328,75.804257,24-03-2022,11:35,11:45,Windy,High,1,Meal,motorcycle,0.0,No,Metropolitian,32
45580,0xd641,AGRRES16DEL01,21.0,4.6,0.000000,0.000000,0.070000,0.070000,16-02-2022,19:55,20:10,Windy,Jam,0,Buffet,motorcycle,1.0,No,Metropolitian,36
45581,0x4f8d,CHENRES08DEL03,30.0,4.9,13.022394,80.242439,13.052394,80.272439,11-03-2022,23:50,24:05:00,Cloudy,Low,1,Drinks,scooter,0.0,No,Metropolitian,16
45582,0x5eee,COIMBRES11DEL01,20.0,4.7,11.001753,76.986241,11.041753,77.026241,07-03-2022,13:35,13:40,Cloudy,High,0,Snack,motorcycle,1.0,No,Metropolitian,26
45583,0x5fb2,RANCHIRES09DEL02,23.0,4.9,23.351058,85.325731,23.431058,85.405731,02-03-2022,17:10,17:15,Fog,Medium,2,Snack,scooter,1.0,No,Metropolitian,36


## 3. Missing-value assessment

We calculate both counts and percentages using the total number of rows as the denominator. This makes the missingness rate comparable across columns.

In [9]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100).round(2)
}).sort_values("missing_count", ascending=False)

display(missing_summary)

,missing_count,missing_percent
Delivery_person_Ratings,1908,4.19
Delivery_person_Age,1854,4.07
Time_Orderd,1731,3.80
City,1200,2.63
multiple_deliveries,993,2.18
Weather_conditions,616,1.35
Road_traffic_density,601,1.32
Festival,228,0.50
ID,0,0.00
Delivery_person_ID,0,0.00


### 3.1 Missingness patterns

This checks whether missing weather and traffic information often occurs together, and whether other fields are also absent in those records. Co-occurrence can reveal a pattern, but it does not prove why values are missing.

In [42]:
# Check rows where both weather and traffic information are missing.
weather_missing = df["Weather_conditions"].isnull()
traffic_missing = df["Road_traffic_density"].isnull()

both_missing = weather_missing & traffic_missing

print("Rows missing both weather and traffic:", both_missing.sum())

# Among those rows, check how many also have age and rating missing.
age_missing = df["Delivery_person_Age"].isnull()
rating_missing = df["Delivery_person_Ratings"].isnull()

also_missing_age_rating = both_missing & age_missing & rating_missing
print("Also missing age and rating:", also_missing_age_rating.sum())

# Check how many of those rows also have City missing.
city_missing = df["City"].isnull()
also_missing_city = both_missing & city_missing

print("Also missing City:", also_missing_city.sum())

Rows missing both weather and traffic: 601
Also missing age and rating: 510
Also missing City: 21


**Interpretation:** missing values are present in several columns. They are retained because the dataset provides no justified imputation rule. No rows are removed merely because one or more fields are missing.

## 4. Duplicate checks

The dataset has a candidate record identifier, `ID`. We check both fully duplicated rows and repeated IDs. A unique ID supports treating each row as a distinct delivery record, although the source does not provide a formal grain definition.

In [43]:
# Check for rows that are completely duplicated.
duplicate_rows = df.duplicated().sum()

# Check whether any ID appears more than once.
duplicate_ids = df["ID"].duplicated().sum()

print("Fully duplicated rows:", duplicate_rows)
print("Repeated ID values:", duplicate_ids)

Fully duplicated rows: 0
Repeated ID values: 0


**Interpretation:** the source contains no fully duplicated rows and no repeated `ID` values. No duplicate removal is required.

## 5. Categorical-column review

We inspect category values and frequencies, and check for leading/trailing whitespace. Rare categories are not automatically errors.

In [12]:
categorical_columns = [
    "Weather_conditions",
    "Road_traffic_density",
    "Type_of_order",
    "Type_of_vehicle",
    "Festival",
    "City"
]

for column in categorical_columns:
    print(f"\\n{column}")
    display(df[column].value_counts(dropna=False))

\nWeather_conditions


Weather_conditions
Fog           7653
Stormy        7584
Cloudy        7533
Sandstorms    7494
Windy         7422
Sunny         7282
NaN            616
Name: count, dtype: int64

\nRoad_traffic_density


Road_traffic_density
Low       15476
Jam       14139
Medium    10945
High       4423
NaN         601
Name: count, dtype: int64

\nType_of_order


Type_of_order
Snack     11530
Meal      11456
Drinks    11321
Buffet    11277
Name: count, dtype: int64

... (2 outputs left out)

\nFestival


Festival
No     44460
Yes      896
NaN      228
Name: count, dtype: int64

\nCity


City
Metropolitian    34087
Urban            10133
NaN               1200
Semi-Urban         164
Name: count, dtype: int64

In [44]:
# Check text columns for spaces at the beginning or end of values.
# We use a simple loop so we can check every text column without repeating code.

whitespace_issues = {}

for column in df.columns:
    if df[column].dtype == "object":
        non_missing = df[column].dropna().astype(str)

        # strip() removes spaces at the beginning and end.
        # If the original value is different, it had extra spaces.
        count = (non_missing != non_missing.str.strip()).sum()

        if count > 0:
            whitespace_issues[column] = count

print("Leading/trailing whitespace counts:", whitespace_issues)

Leading/trailing whitespace counts: {}


**Interpretation and decision:**
- No leading/trailing whitespace was detected in the inspected text columns.
- `Metropolitian` may be a misspelling, but it is retained because the intended replacement is not verified.
- `Jam` is retained because its meaning is unclear without a data dictionary.
- No categorical labels are changed.

## 6. Numerical-column review

The following checks summarize distributions and flag values for review. They do not automatically classify rare or extreme values as invalid.

In [45]:
numeric_columns = df.select_dtypes(include="number").columns.tolist()
display(df[numeric_columns].describe())

,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Vehicle_condition,multiple_deliveries,Time_taken (min)
count,43730.000000,43676.000000,45584.000000,45584.000000,45584.000000,45584.000000,45584.000000,44591.000000,45584.000000
mean,29.566911,4.633774,17.017948,70.229684,17.465480,70.844161,1.023385,0.744635,26.293963
std,5.815064,0.334744,8.185674,22.885575,7.335562,21.120578,0.839055,0.572510,9.384298
min,15.000000,1.000000,-30.905562,-88.366217,0.010000,0.010000,0.000000,0.000000,10.000000
25%,25.000000,4.500000,12.933284,73.170000,12.988453,73.280000,0.000000,0.000000,19.000000
50%,30.000000,4.700000,18.551440,75.897963,18.633934,76.002574,1.000000,1.000000,26.000000
75%,35.000000,4.900000,22.728163,78.044095,22.785049,78.107044,2.000000,1.000000,32.000000
max,50.000000,6.000000,30.914057,88.433452,31.054057,88.563452,3.000000,3.000000,54.000000


In [46]:
print("Delivery person age frequencies:")
display(df["Delivery_person_Age"].value_counts(dropna=False).sort_index())

age_15_count = (df["Delivery_person_Age"] == 15).sum()
print("Records with age 15:", age_15_count)

Delivery person age frequencies:


Delivery_person_Age
15.0      38
20.0    2136
21.0    2153
22.0    2194
23.0    2086
24.0    2210
25.0    2174
26.0    2159
27.0    2150
28.0    2179
29.0    2191
30.0    2226
31.0    2120
32.0    2201
33.0    2186
34.0    2165
35.0    2261
36.0    2260
37.0    2226
38.0    2218
39.0    2144
50.0      53
NaN     1854
Name: count, dtype: int64

Records with age 15: 38


In [48]:
print("Delivery person rating frequencies:")
display(df["Delivery_person_Ratings"].value_counts(dropna=False).sort_index())

# Count the records with a rating of 6.
rating_6_count = (df["Delivery_person_Ratings"] == 6).sum()

# Calculate the percentage using only rows that have a rating.
ratings_without_missing = df["Delivery_person_Ratings"].count()
rating_6_percent = (rating_6_count / ratings_without_missing) * 100

print("Records with rating 6:", rating_6_count)
print("Percentage of non-missing ratings 6:", round(rating_6_percent, 3), "%")

Delivery person rating frequencies:


Delivery_person_Ratings
1.0      38
2.5      20
2.6      22
2.7      22
2.8      19
2.9      19
3.0       6
3.1      29
3.2      29
3.3      25
3.4      32
3.5     249
3.6     207
3.7     225
3.8     228
3.9     197
4.0    1077
4.1    1430
4.2    1418
4.3    1409
4.4    1360
4.5    3302
4.6    6938
4.7    7140
4.8    7146
4.9    7040
5.0    3996
6.0      53
NaN    1908
Name: count, dtype: int64

Records with rating 6: 53
Percentage of non-missing ratings 6: 0.121 %


In [49]:
# Inspect the values in these three numerical columns.
columns_to_check = ["Vehicle_condition", "multiple_deliveries", "Time_taken (min)"]

for column in columns_to_check:
    print("\\n", column)
    display(df[column].value_counts(dropna=False).sort_index())

\n Vehicle_condition


Vehicle_condition
0    15005
1    15028
2    15031
3      520
Name: count, dtype: int64

\n multiple_deliveries


multiple_deliveries
0.0    14094
1.0    28151
2.0     1985
3.0      361
NaN      993
Name: count, dtype: int64

\n Time_taken (min)


Time_taken (min)
10     750
11     757
12     746
13     716
14     739
15    1810
16    1706
17    1696
18    1765
19    1824
20    1640
21    1600
22    1626
23    1643
24    1679
25    2050
26    2121
27    1976
28    1965
29    1956
30    1217
31    1213
32    1123
33    1258
34    1171
35     832
36     852
37     828
38     887
39     847
40     554
41     553
42     561
43     567
44     553
45     241
46     274
47     295
48     277
49     280
50      72
51      94
52      79
53     100
54      91
Name: count, dtype: int64

**Interpretation and decision:**
- Age 15 occurs 38 times. The value is unusual relative to the observed age distribution, but no official minimum-age rule is supplied; retain and flag it.
- Rating 6 occurs 53 times (about 0.121% of non-missing ratings). The apparent scale is concentrated at 1–5, but the official scale is undocumented; retain and flag it.
- `Vehicle_condition` contains 0, 1, 2 and 3; the code meanings are unknown. Retain all codes.
- `multiple_deliveries` contains 0, 1, 2 and 3, plus missing values. Retain observed values.
- `Time_taken (min)` ranges from 10 to 54 with no missing values. The observed range alone does not establish invalidity; retain all values.

## 7. Order-date validation

The source dates appear in day-month-year format. We parse using that explicit format and check for failures. We also inspect calendar coverage; a date with no records is a coverage gap, not a row that can safely be reconstructed.

In [50]:
# Convert Order_Date using the day-month-year format found in the source.
# errors="coerce" turns any date that cannot be read into a missing value.
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

invalid_date_count = df["Order_Date"].isnull().sum()

print("Invalid or unparsed dates:", invalid_date_count)
print("First date:", df["Order_Date"].min())
print("Last date:", df["Order_Date"].max())
print("Different dates with records:", df["Order_Date"].nunique())

Invalid or unparsed dates: 0
First date: 2022-02-11 00:00:00
Last date: 2022-04-06 00:00:00
Different dates with records: 44


In [51]:
# Check every calendar day between the first and last dates.
# If a date is not found in the dataset, add it to the missing dates list.

from datetime import timedelta

start_date = df["Order_Date"].min()
end_date = df["Order_Date"].max()

dates_present = df["Order_Date"].dropna().unique().tolist()
missing_calendar_dates = []

current_date = start_date

while current_date <= end_date:
    if current_date not in dates_present:
        missing_calendar_dates.append(current_date)

    current_date = current_date + timedelta(days=1)

print("Calendar days in date range:", (end_date - start_date).days + 1)
print("Dates with records:", len(dates_present))
print("Dates with no records:", len(missing_calendar_dates))

display(pd.DataFrame({
    "date_without_records": missing_calendar_dates
}))

Calendar days in date range: 55
Dates with records: 44
Dates with no records: 11


,date_without_records
0,2022-02-19
1,2022-02-20
2,2022-02-21
3,2022-02-22
4,2022-02-23
5,2022-02-24
6,2022-02-25
7,2022-02-26
8,2022-02-27
9,2022-02-28


**Interpretation:** all source date strings parse successfully with `%d-%m-%Y`. There are 44 dates with records across the range 2022-02-11 to 2022-04-06; 11 calendar dates have no records. This is documented as a coverage limitation. No dates or rows are fabricated.

## 8. Geographic-coordinate review

We check negative restaurant coordinates, zero restaurant-coordinate pairs, and unusual delivery-coordinate pairs. Without verified location metadata, we do not flip signs, replace zeros, or delete records.

In [52]:
# Check the four coordinate columns.
coordinate_columns = [
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude"
]

display(df[coordinate_columns].describe())

# Count negative values in each coordinate column.
for column in coordinate_columns:
    negative_count = (df[column] < 0).sum()
    print(column, "- negative values:", negative_count)

,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude
count,45584.000000,45584.000000,45584.000000,45584.000000
mean,17.017948,70.229684,17.465480,70.844161
std,8.185674,22.885575,7.335562,21.120578
min,-30.905562,-88.366217,0.010000,0.010000
25%,12.933284,73.170000,12.988453,73.280000
50%,18.551440,75.897963,18.633934,76.002574
75%,22.728163,78.044095,22.785049,78.107044
max,30.914057,88.433452,31.054057,88.563452


Restaurant_latitude - negative values: 431
Restaurant_longitude - negative values: 162
Delivery_location_latitude - negative values: 0
Delivery_location_longitude - negative values: 0


In [53]:
# Check restaurant coordinates that are negative.
negative_restaurant_latitude = df["Restaurant_latitude"] < 0
both_restaurant_coordinates_negative = (
    (df["Restaurant_latitude"] < 0) &
    (df["Restaurant_longitude"] < 0)
)
negative_latitude_positive_longitude = (
    (df["Restaurant_latitude"] < 0) &
    (df["Restaurant_longitude"] > 0)
)

print("Negative restaurant latitude:", negative_restaurant_latitude.sum())
print("Both restaurant coordinates negative:",
      both_restaurant_coordinates_negative.sum())
print("Negative latitude and positive longitude:",
      negative_latitude_positive_longitude.sum())

Negative restaurant latitude: 431
Both restaurant coordinates negative: 162
Negative latitude and positive longitude: 269


In [54]:
# Compare the absolute coordinate values for rows with negative restaurant latitude.
# This helps us see whether the values look similar in size to delivery coordinates.

negative_restaurant_rows = df[negative_restaurant_latitude]

latitude_difference = (
    negative_restaurant_rows["Restaurant_latitude"].abs() -
    negative_restaurant_rows["Delivery_location_latitude"].abs()
).abs()

longitude_difference = (
    negative_restaurant_rows["Restaurant_longitude"].abs() -
    negative_restaurant_rows["Delivery_location_longitude"].abs()
).abs()

print("Average latitude difference:", round(latitude_difference.mean(), 6))
print("Average longitude difference:", round(longitude_difference.mean(), 6))

Average latitude difference: 0.064014
Average longitude difference: 0.064014


In [55]:
# Find rows where both restaurant coordinates are zero.
zero_restaurant_pair = (
    (df["Restaurant_latitude"] == 0) &
    (df["Restaurant_longitude"] == 0)
)

# Find rows where both delivery coordinates are 0.01.
delivery_001_pair = (
    (df["Delivery_location_latitude"] == 0.01) &
    (df["Delivery_location_longitude"] == 0.01)
)

print("Restaurant coordinates equal to (0, 0):",
      zero_restaurant_pair.sum())

print("Among these, delivery coordinates equal to (0.01, 0.01):",
      (zero_restaurant_pair & delivery_001_pair).sum())

# Check whether delivery latitude and longitude are equal in these rows.
zero_restaurant_rows = df[zero_restaurant_pair]
same_delivery_coordinates = (
    zero_restaurant_rows["Delivery_location_latitude"] ==
    zero_restaurant_rows["Delivery_location_longitude"]
)

print("Rows where delivery latitude equals longitude:",
      same_delivery_coordinates.sum(), "out of", len(zero_restaurant_rows))

Restaurant coordinates equal to (0, 0): 3640
Among these, delivery coordinates equal to (0.01, 0.01): 327
Rows where delivery latitude equals longitude: 3640 out of 3640


In [56]:
# Compare the percentage of zero restaurant coordinates within each city.
# We calculate the percentage separately for each city.

cities = df["City"].dropna().unique().tolist()

for city in cities:
    city_rows = df["City"] == city
    total_in_city = city_rows.sum()

    zero_coordinates_in_city = (city_rows & zero_restaurant_pair).sum()

    percentage = (zero_coordinates_in_city / total_in_city) * 100

    print(city)
    print("Total records:", total_in_city)
    print("Records with zero restaurant coordinates:", zero_coordinates_in_city)
    print("Percentage:", round(percentage, 2), "%")
    print("--------------------------")

# Also check rows where the City value is missing.
missing_city_rows = df["City"].isnull()
total_missing_city = missing_city_rows.sum()
zero_coordinates_missing_city = (missing_city_rows & zero_restaurant_pair).sum()

print("City is missing")
print("Total records:", total_missing_city)
print("Zero restaurant coordinates:", zero_coordinates_missing_city)
if total_missing_city > 0:
    print("Percentage:", round(zero_coordinates_missing_city / total_missing_city * 100, 2), "%")

Metropolitian
Total records: 34087
Records with zero restaurant coordinates: 2682
Percentage: 7.87 %
--------------------------
Urban
Total records: 10133
Records with zero restaurant coordinates: 857
Percentage: 8.46 %
--------------------------
Semi-Urban
Total records: 164
Records with zero restaurant coordinates: 15
Percentage: 9.15 %
--------------------------
City is missing
Total records: 1200
Zero restaurant coordinates: 86
Percentage: 7.17 %


**Interpretation and decision:**
- 431 records have negative restaurant latitude; 162 of these have both restaurant coordinates negative, and 269 have negative latitude with positive longitude.
- The negative-coordinate records show close magnitudes to delivery coordinates in many cases, suggesting a systematic sign/geography issue, but the correct restaurant coordinates cannot be reconstructed reliably.
- 3,640 records have restaurant coordinates `(0, 0)`. In all 3,640, delivery latitude equals delivery longitude; 327 also have delivery coordinates `(0.01, 0.01)`. These are suspicious patterns, not proof of the intended values.
- The within-city rates are calculated using each city's own row count as denominator.
- All coordinate values are retained and flagged. Do not flip signs or replace zeros without source confirmation.

## 9. Order-time and pickup-time review

The time columns contain multiple representations. We quantify them and check relationships that can be validated without forcing a potentially incorrect conversion.

In [57]:
# Inspect the two time columns.
time_columns = ["Time_Orderd", "Time_Order_picked"]

for column in time_columns:
    print("\\n", column)
    print("Missing values:", df[column].isnull().sum())

    # Count values written in the usual HH:MM style.
    colon_count = 0
    numeric_count = 0

    for value in df[column].dropna():
        value = str(value)

        if ":" in value:
            colon_count = colon_count + 1
        else:
            # Try to turn the value into a number.
            try:
                float(value)
                numeric_count = numeric_count + 1
            except ValueError:
                pass

    print("Values containing a colon:", colon_count)
    print("Numeric-looking values:", numeric_count)

\n Time_Orderd
Missing values: 1731
Values containing a colon: 39785
Numeric-looking values: 4068
\n Time_Order_picked
Missing values: 0
Values containing a colon: 41457
Numeric-looking values: 4127


In [26]:
pickup_after_24 = df["Time_Order_picked"].astype(str).str.startswith("24:")
print("Pickup values beginning with 24:", int(pickup_after_24.sum()))
display(df.loc[pickup_after_24, "Time_Order_picked"].value_counts())
print("Dates containing a pickup time beginning with 24:",
      df.loc[pickup_after_24, "Order_Date"].nunique())
print("Order-time values paired with pickup times beginning with 24:")
display(df.loc[pickup_after_24, "Time_Orderd"].value_counts(dropna=False))

Pickup values beginning with 24: 880


Time_Order_picked
24:05:00    435
24:10:00    280
24:15:00    165
Name: count, dtype: int64

Dates containing a pickup time beginning with 24: 44
Order-time values paired with pickup times beginning with 24:


Time_Orderd
1        430
23:55    272
23:50    139
NaN       39
Name: count, dtype: int64

In [59]:
# Compare order and pickup times when both use ordinary HH:MM format.
# We convert them to minutes after midnight using a simple function.

def time_to_minutes(value):
    if pd.isnull(value):
        return None

    value = str(value)

    if ":" not in value:
        return None

    parts = value.split(":")
    if len(parts) < 2:
        return None

    hour = int(parts[0])
    minute = int(parts[1])

    if hour < 0 or hour > 23 or minute < 0 or minute > 59:
        return None

    return hour * 60 + minute


time_differences = []

for index in range(len(df)):
    order_time = time_to_minutes(df.loc[index, "Time_Orderd"])
    pickup_time = time_to_minutes(df.loc[index, "Time_Order_picked"])

    if order_time is not None and pickup_time is not None:
        difference = pickup_time - order_time
        time_differences.append(difference)

print("Rows with both times in ordinary HH:MM format:", len(time_differences))
print("Pickup earlier than order:", sum(value < 0 for value in time_differences))
print("Pickup later by 5 minutes:", time_differences.count(5))
print("Pickup later by 10 minutes:", time_differences.count(10))
print("Pickup later by 15 minutes:", time_differences.count(15))

Rows with both times in ordinary HH:MM format: 35407
Pickup earlier than order: 0
Pickup later by 5 minutes: 12022
Pickup later by 10 minutes: 11753
Pickup later by 15 minutes: 11632


In [28]:
# Demonstrate how a decimal fraction of a day maps to clock time.
# Rounding total minutes first avoids displaying values such as 10:60.
decimal_examples = pd.Series([0.458333333, 0.75, 0.958333333, 0.833333333],
                             name="decimal_fraction")
total_minutes = (decimal_examples * 24 * 60).round().astype(int)
hours = total_minutes // 60
minutes = total_minutes % 60

display(pd.DataFrame({
    "original_value": decimal_examples,
    "interpreted_time_example": [
        f"{hour:02d}:{minute:02d}" for hour, minute in zip(hours, minutes)
    ]
}))

,original_value,interpreted_time_example
0,0.458333,11:00
1,0.750000,18:00
2,0.958333,23:00
3,0.833333,20:00


**Interpretation and decision:**
- `Time_Orderd` contains 4,068 numeric-like values; `Time_Order_picked` contains 4,127. These values appear to be fractions of a 24-hour day, but the source does not document this encoding.
- `Time_Order_picked` contains 880 values beginning with `24:` (`24:05:00`, `24:10:00`, `24:15:00`) across 44 dates. These may represent times shortly after midnight.
- For 35,407 rows where both values are ordinary `HH:MM` strings, pickup is later by 5, 10 or 15 minutes. This supports the general relationship between the columns but does not resolve every mixed-format value.
- The numeric value `1` may represent the end of the day; it is not converted because the rollover convention is undocumented.
- Both time columns are retained exactly as supplied. Do not use ordinary datetime conversion on the mixed values until the encoding and date rollover are confirmed.

## 10. Cleaning decisions and export

The cleaned dataset is deliberately conservative. Only `Order_Date` is standardized after successful parsing. All other source values, including missing values and flagged anomalies, are preserved.

In [29]:
df_clean = df_raw.copy()
df_clean["Order_Date"] = pd.to_datetime(
    df_clean["Order_Date"],
    format="%d-%m-%Y",
    errors="raise"
)

CLEANED_DATA_PATH = PROCESSED_DIR / "cleaned_food_delivery.csv"
df_clean.to_csv(CLEANED_DATA_PATH, index=False)

print("Cleaned dataset saved to:", CLEANED_DATA_PATH)
print("Original shape:", df_raw.shape)
print("Cleaned shape:", df_clean.shape)

Cleaned dataset saved to: C:\Users\DELL\Documents\a project intern\data\processed\cleaned_food_delivery.csv
Original shape: (45584, 20)
Cleaned shape: (45584, 20)


### 10.1 Create the cleaning log

The log records the issue, affected column/records, action, and reason. It distinguishes actual modifications from anomalies that were deliberately retained.

In [30]:
cleaning_log_rows = []

for column, count in df_raw.isna().sum().items():
    if count > 0:
        cleaning_log_rows.append({
            "issue": "Missing values",
            "affected_column": column,
            "affected_rows": int(count),
            "action_taken": "Retained missing values; no imputation or row deletion",
            "reason": "No documented rule supports a reliable replacement."
        })

cleaning_log_rows.extend([
    {
        "issue": "Date format",
        "affected_column": "Order_Date",
        "affected_rows": len(df_raw),
        "action_taken": "Parsed day-month-year dates and exported in ISO date format",
        "reason": "All source dates parsed successfully using %d-%m-%Y."
    },
    {
        "issue": "Incomplete calendar coverage",
        "affected_column": "Order_Date",
        "affected_rows": f"{len(missing_calendar_dates)} calendar dates absent",
        "action_taken": "Documented; no dates or records fabricated",
        "reason": "The source does not explain the gaps."
    },
    {
        "issue": "Potentially unusual age",
        "affected_column": "Delivery_person_Age",
        "affected_rows": int(df_raw["Delivery_person_Age"].eq(15).sum()),
        "action_taken": "Retained and flagged",
        "reason": "No authoritative minimum-age rule was supplied."
    },
    {
        "issue": "Rating outside apparent pattern",
        "affected_column": "Delivery_person_Ratings",
        "affected_rows": int(df_raw["Delivery_person_Ratings"].eq(6).sum()),
        "action_taken": "Retained and flagged",
        "reason": "The official rating scale is undocumented."
    },
    {
        "issue": "Negative restaurant coordinates",
        "affected_column": "Restaurant_latitude, Restaurant_longitude",
        "affected_rows": int(df_raw["Restaurant_latitude"].lt(0).sum()),
        "action_taken": "Retained and flagged; no sign changes",
        "reason": "Correct coordinates cannot be verified from the supplied fields."
    },
    {
        "issue": "Zero restaurant coordinate pair",
        "affected_column": "Restaurant_latitude, Restaurant_longitude",
        "affected_rows": int(zero_restaurant_pair.sum()),
        "action_taken": "Retained and flagged",
        "reason": "Possible placeholder values, but no confirmed replacements."
    },
    {
        "issue": "Suspicious delivery coordinate pair",
        "affected_column": "Delivery_location_latitude, Delivery_location_longitude",
        "affected_rows": int((zero_restaurant_pair & delivery_001_pair).sum()),
        "action_taken": "Retained and flagged",
        "reason": "The pair (0.01, 0.01) is suspicious but not verified as an error."
    },
    {
        "issue": "Mixed time representations",
        "affected_column": "Time_Orderd, Time_Order_picked",
        "affected_rows": "4068 numeric-like order times; 4127 numeric-like pickup times",
        "action_taken": "Retained original representations; no conversion",
        "reason": "Mixed HH:MM and decimal-fraction formats require source confirmation."
    },
    {
        "issue": "Pickup times beyond 24:00",
        "affected_column": "Time_Order_picked",
        "affected_rows": int(pickup_after_24.sum()),
        "action_taken": "Retained original values",
        "reason": "Likely day rollover, but the convention is undocumented."
    },
    {
        "issue": "Unverified categorical labels",
        "affected_column": "City, Road_traffic_density",
        "affected_rows": "Metropolitian and Jam labels retained",
        "action_taken": "No renaming",
        "reason": "Intended meanings cannot be verified without a data dictionary."
    },
    {
        "issue": "Unverified coded category values",
        "affected_column": "Vehicle_condition",
        "affected_rows": int(df_raw["Vehicle_condition"].eq(3).sum()),
        "action_taken": "Retained all observed codes",
        "reason": "Code meanings and valid range are undocumented."
    }
])

cleaning_log = pd.DataFrame(
    cleaning_log_rows,
    columns=["issue", "affected_column", "affected_rows", "action_taken", "reason"]
)

CLEANING_LOG_PATH = DOCS_DIR / "cleaning_log.csv"
cleaning_log.to_csv(CLEANING_LOG_PATH, index=False)

display(cleaning_log)
print("Cleaning log saved to:", CLEANING_LOG_PATH)

,issue,affected_column,affected_rows,action_taken,reason
0,Missing values,Delivery_person_Age,1854,Retained missing values; no imputation or row ...,No documented rule supports a reliable replace...
1,Missing values,Delivery_person_Ratings,1908,Retained missing values; no imputation or row ...,No documented rule supports a reliable replace...
2,Missing values,Time_Orderd,1731,Retained missing values; no imputation or row ...,No documented rule supports a reliable replace...
3,Missing values,Weather_conditions,616,Retained missing values; no imputation or row ...,No documented rule supports a reliable replace...
4,Missing values,Road_traffic_density,601,Retained missing values; no imputation or row ...,No documented rule supports a reliable replace...
5,Missing values,multiple_deliveries,993,Retained missing values; no imputation or row ...,No documented rule supports a reliable replace...
6,Missing values,Festival,228,Retained missing values; no imputation or row ...,No documented rule supports a reliable replace...
7,Missing values,City,1200,Retained missing values; no imputation or row ...,No documented rule supports a reliable replace...
8,Date format,Order_Date,45584,Parsed day-month-year dates and exported in IS...,All source dates parsed successfully using %d-...
9,Incomplete calendar coverage,Order_Date,11 calendar dates absent,Documented; no dates or records fabricated,The source does not explain the gaps.


Cleaning log saved to: C:\Users\DELL\Documents\a project intern\docs\cleaning_log.csv


## 11. Final verification

We verify that the exported file can be read, has the same number of rows and columns, retains the source missing-value counts, and contains no newly duplicated IDs. This confirms that the conservative cleaning process did not silently remove records.

In [31]:
reloaded_clean = pd.read_csv(CLEANED_DATA_PATH)

verification = {
    "same_shape": reloaded_clean.shape == df_raw.shape,
    "same_columns": reloaded_clean.columns.tolist() == df_raw.columns.tolist(),
    "same_ID_count": reloaded_clean["ID"].nunique() == df_raw["ID"].nunique(),
    "same_missing_counts": reloaded_clean.isna().sum().equals(df_raw.isna().sum()),
    "no_duplicate_IDs": reloaded_clean["ID"].duplicated().sum() == 0,
    "date_format_exported_as_ISO": pd.to_datetime(
        reloaded_clean["Order_Date"], format="%Y-%m-%d", errors="coerce"
    ).notna().all()
}

display(pd.Series(verification, name="passed"))
assert all(verification.values()), "At least one final verification check failed."

print("Final verification passed.")

same_shape                     True
same_columns                   True
same_ID_count                  True
same_missing_counts            True
no_duplicate_IDs               True
date_format_exported_as_ISO    True
Name: passed, dtype: bool

Final verification passed.


## 12. Week 1 conclusion

The dataset contains 45,584 rows and 20 columns. The review identified missing values, suspicious age/rating values, geographic-coordinate anomalies, mixed time encodings, and gaps in calendar coverage. Duplicate checks found no duplicate rows or IDs.

The cleaned CSV preserves all records and all uncertain source values. The only applied transformation is the verified conversion of `Order_Date` to a standard date representation. The cleaning log records both the issues and the reasons for retaining values that cannot yet be corrected safely.

### Questions to confirm with the dataset owner or mentor
1. What are the valid meanings/ranges for `Vehicle_condition` and `multiple_deliveries`?
2. Is the rating scale officially 1–5, and how should rating 6 be interpreted?
3. Are negative and zero restaurant coordinates known placeholders or sign errors?
4. Are decimal time values fractions of a day, and how should `24:xx` be associated with `Order_Date`?
5. Is `Metropolitian` the intended spelling, and what does `Jam` mean in the traffic-density field?